# 02 - Feature Engineering

## Objective

This notebook prepares the cleaned German electricity market dataset for analysis.

The cleaned dataset contains hourly electricity generation, consumption, and wholesale price data from 2021 to 2025.

Additional variables will be created to support the subsequent exploratory analysis, SQL analysis, and Power BI dashboard.

The feature engineering stage includes:

- Time-based features
- Total electricity generation
- Renewable generation
- Conventional generation
- Renewable share
- Electricity price indicators

In [3]:
import pandas as pd
import numpy as np

In [5]:
electricity_data = pd.read_csv(
    "../data/processed/electricity_data_clean.csv",
    parse_dates=["timestamp"]
)

In [7]:
electricity_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 26 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   timestamp                   43824 non-null  datetime64[ns]
 1   biomass                     43824 non-null  float64       
 2   hydro                       43824 non-null  float64       
 3   wind_offshore               43824 non-null  float64       
 4   wind_onshore                43824 non-null  float64       
 5   solar                       43824 non-null  float64       
 6   other_renewables            43824 non-null  float64       
 7   nuclear                     43824 non-null  float64       
 8   lignite                     43824 non-null  float64       
 9   hard_coal                   43824 non-null  float64       
 10  gas                         43824 non-null  float64       
 11  pumped_storage_generation   43824 non-null  float64   

## 1. Time Features

Time-based features are extracted from the hourly timestamp to analyse how electricity generation, demand, and wholesale prices vary over time.

The following variables are created:

- **Year** - to compare developments between 2021 and 2025.
- **Month** - to identify seasonal patterns.
- **Hour** - to analyse patterns throughout the day.
- **Day of week** - to compare electricity-market behaviour across different days of the week.

In [13]:
electricity_data["year"] = electricity_data["timestamp"].dt.year #Because timestamp is a datetime column, pandas gives us the .dt accessor.
electricity_data["month"] = electricity_data["timestamp"].dt.month
electricity_data["hour"] = electricity_data["timestamp"].dt.hour
electricity_data["day_of_week"] = electricity_data["timestamp"].dt.day_name()

In [15]:
electricity_data[
    ["timestamp", "year", "month", "hour", "day_of_week"]
].head(10)

,timestamp,year,month,hour,day_of_week
0,2021-01-01 00:00:00,2021,1,0,Friday
1,2021-01-01 01:00:00,2021,1,1,Friday
2,2021-01-01 02:00:00,2021,1,2,Friday
3,2021-01-01 03:00:00,2021,1,3,Friday
4,2021-01-01 04:00:00,2021,1,4,Friday
5,2021-01-01 05:00:00,2021,1,5,Friday
6,2021-01-01 06:00:00,2021,1,6,Friday
7,2021-01-01 07:00:00,2021,1,7,Friday
8,2021-01-01 08:00:00,2021,1,8,Friday
9,2021-01-01 09:00:00,2021,1,9,Friday


## 2. Generation Features

### 2.1 Renewable Generation

Renewable generation represents the total hourly electricity generation from renewable energy sources in the dataset.

It is calculated as the sum of biomass, hydropower, offshore wind, onshore wind, solar, and other renewable generation.

In [20]:
renewable_columns = [
    "biomass",
    "hydro",
    "wind_offshore",
    "wind_onshore",
    "solar",
    "other_renewables"
] #renewable_columns is simply a list telling Python which columns belong to our renewable category.

electricity_data["renewable_generation"] = electricity_data[
    renewable_columns
].sum(axis=1) #Add across the columns for each row/hour.

In [22]:
electricity_data[
    renewable_columns + ["renewable_generation"]
].head()

,biomass,hydro,wind_offshore,wind_onshore,solar,other_renewables,renewable_generation
0,4481.00,1203.00,383.00,3928.25,1.0,213.00,10209.25
1,4453.00,1192.75,394.50,3528.25,1.0,213.25,9782.75
2,4440.50,1161.00,305.25,3198.00,1.0,216.50,9322.25
3,4424.75,1177.75,319.25,2768.75,1.0,218.00,8909.50
4,4430.75,1153.25,296.25,2462.25,1.0,221.50,8565.00


### 2.2 Total Generation

Total generation represents the sum of all electricity generation recorded for each hour.

Pumped-storage generation is included in total generation because it supplies electricity to the grid during discharge. However, it is treated separately from renewable and conventional energy sources because it represents stored electricity rather than a primary energy source.

In [25]:
generation_columns = [
    "biomass",
    "hydro",
    "wind_offshore",
    "wind_onshore",
    "solar",
    "other_renewables",
    "nuclear",
    "lignite",
    "hard_coal",
    "gas",
    "pumped_storage_generation",
    "other_conventional"
]

electricity_data["total_generation"] = electricity_data[
    generation_columns
].sum(axis=1)

In [27]:
electricity_data[
    ["timestamp", "renewable_generation", "total_generation"]
].head()

,timestamp,renewable_generation,total_generation
0,2021-01-01 00:00:00,10209.25,42314.00
1,2021-01-01 01:00:00,9782.75,41423.00
2,2021-01-01 02:00:00,9322.25,40690.50
3,2021-01-01 03:00:00,8909.50,40273.25
4,2021-01-01 04:00:00,8565.00,39784.25


### 2.3 Renewable Share

Renewable share represents the percentage of total hourly electricity generation that comes from renewable energy sources.

This makes it easier to compare the importance of renewable electricity across different hours and years, even when total electricity generation changes.

In [30]:
electricity_data["renewable_share"] = (
    electricity_data["renewable_generation"]
    / electricity_data["total_generation"]
    * 100
)

In [32]:
electricity_data[
    ["timestamp", "renewable_generation",
     "total_generation", "renewable_share"]
].head()

,timestamp,renewable_generation,total_generation,renewable_share
0,2021-01-01 00:00:00,10209.25,42314.00,24.127357
1,2021-01-01 01:00:00,9782.75,41423.00,23.616711
2,2021-01-01 02:00:00,9322.25,40690.50,22.910139
3,2021-01-01 03:00:00,8909.50,40273.25,22.122625
4,2021-01-01 04:00:00,8565.00,39784.25,21.528620


### 2.4 Conventional Generation

Conventional generation combines the electricity generated from nuclear, lignite, hard coal, gas, and other conventional sources.

Pumped-storage generation is excluded because it represents stored electricity being returned to the grid rather than electricity produced from a primary energy source.

In [35]:
conventional_columns = [
    "nuclear",
    "lignite",
    "hard_coal",
    "gas",
    "other_conventional"
]

electricity_data["conventional_generation"] = electricity_data[
    conventional_columns
].sum(axis=1)

In [37]:
electricity_data[
    conventional_columns + ["conventional_generation"]
].head()

,nuclear,lignite,hard_coal,gas,other_conventional,conventional_generation
0,8144.75,11608.50,3443.75,6923.0,1637.5,31757.50
1,8150.25,11602.75,3044.75,6688.0,1636.0,31121.75
2,8156.50,11758.50,3067.25,6586.0,1630.0,31198.25
3,8153.75,12337.50,2852.50,6396.5,1621.5,31361.75
4,8150.50,12395.00,2713.25,6333.0,1621.5,31213.25


## 3. Price Features

### 3.1 Negative Electricity Prices

A negative-price indicator is created to identify hours in which the German/Luxembourg day-ahead wholesale electricity price falls below €0/MWh.

This feature will later be used to investigate whether negative-price hours are associated with particular conditions in renewable generation, electricity demand, and residual load.

In [40]:
electricity_data["negative_price"] = (
    electricity_data["price_germany"] < 0
)

In [42]:
electricity_data["negative_price"].value_counts()

negative_price
False    42285
True      1539
Name: count, dtype: int64

In [44]:
electricity_data.loc[
    electricity_data["negative_price"],
    ["timestamp", "price_germany", "renewable_share", "load", "residual_load"]
].head(10)

,timestamp,price_germany,renewable_share,load,residual_load
888,2021-02-07 00:00:00,-1.24,64.686129,51755.50,18249.75
889,2021-02-07 01:00:00,-1.59,65.034280,49829.50,16356.75
890,2021-02-07 02:00:00,-0.02,65.129607,48811.25,15363.25
891,2021-02-07 03:00:00,-0.04,64.829130,48242.75,15673.75
892,2021-02-07 04:00:00,-2.01,64.722975,48386.75,15830.75
893,2021-02-07 05:00:00,-0.09,64.454897,48262.50,15735.75
894,2021-02-07 06:00:00,-3.84,64.513303,47604.75,14853.00
895,2021-02-07 07:00:00,-0.09,64.202785,49689.00,16719.75
896,2021-02-07 08:00:00,-2.38,63.198793,53051.00,20242.25
1692,2021-03-12 12:00:00,-5.09,77.721904,73338.25,12986.25


## 4. Feature Validation

Before exporting the feature-engineered dataset, the newly created variables are validated to ensure that the calculations are consistent and do not introduce unexpected missing or invalid values.

The validation includes:

- Checking the new features for missing values.
- Verifying that renewable share remains within a valid range.
- Confirming that total generation equals renewable, conventional, and pumped-storage generation combined.
- Checking the distribution of the negative-price indicator.

In [47]:
new_features = [
    "year",
    "month",
    "hour",
    "day_of_week",
    "renewable_generation",
    "conventional_generation",
    "total_generation",
    "renewable_share",
    "negative_price"
]

electricity_data[new_features].isna().sum()

year                       0
month                      0
hour                       0
day_of_week                0
renewable_generation       0
conventional_generation    0
total_generation           0
renewable_share            0
negative_price             0
dtype: int64

In [49]:
electricity_data["renewable_share"].agg(["min", "max"])

min    12.220021
max    91.067983
Name: renewable_share, dtype: float64

In [51]:
generation_difference = (
    electricity_data["total_generation"]
    - (
        electricity_data["renewable_generation"]
        + electricity_data["conventional_generation"]
        + electricity_data["pumped_storage_generation"]
    )
)

generation_difference.abs().max()

2.1827872842550278e-11

## 5. Export Feature-Engineered Dataset

After creating and validating the additional analytical features, the final dataset is exported for use in the exploratory analysis, SQL analysis, and Power BI dashboard.

The exported dataset contains the original cleaned electricity-market variables together with the newly created time, generation, and price features.

In [54]:
electricity_data.to_csv(
    "../data/processed/electricity_data_features.csv",
    index=False
)